# 06 - Inference & Export
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Load trained model checkpoints
2. Single URL 4-class inference pipeline
3. Batch inference pipeline with timing benchmark
4. Build Counting Bloom Filter from all threat-class URLs (phishing + malware + spam)
5. Export Bloom Filter, fusion model, and XGBoost for production

In [ ]:
import sys
import os
import pickle
import json
import logging
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizer

from ml.src.data.data_loader import load_cic_bell_dns2021
from ml.src.features.url_features import extract_url_features

logging.basicConfig(level=logging.WARNING)

CLASS_NAMES    = ["benign", "phishing", "malware", "spam"]
NUM_CLASSES    = 4
CHECKPOINT_DIR = Path(PROJECT_ROOT) / "ml" / "checkpoints"
DATA_DIR       = Path(PROJECT_ROOT) / "data" / "raw"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 6.1 Load Model Components

In [ ]:
## 6.1 Load Model Components

# ── Compact model definitions ────────────────────────────────────────────────
class AttentionLayer(nn.Module):
    def __init__(self, h): super().__init__(); self.a = nn.Linear(h, 1)
    def forward(self, x): return torch.sum(torch.softmax(self.a(x), dim=1) * x, dim=1)

class NLPBranch(nn.Module):
    def __init__(self, out=128, lstm_h=256, layers=2, dropout=0.3, freeze=True):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        if freeze:
            for p in self.bert.parameters(): p.requires_grad = False
        bh = self.bert.config.hidden_size
        self.lstm = nn.LSTM(bh, lstm_h, layers, batch_first=True, bidirectional=True,
                            dropout=dropout if layers > 1 else 0)
        self.attn = AttentionLayer(lstm_h * 2)
        self.fc = nn.Linear(lstm_h * 2, out)
        self.drop = nn.Dropout(dropout)
    def forward(self, ids, mask):
        x = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state
        x, _ = self.lstm(x)
        return self.fc(self.drop(self.attn(x)))

class MLPBranch(nn.Module):
    def __init__(self, in_dim=23, hidden=None, out=64, drop=0.3):
        super().__init__()
        if hidden is None: hidden = [128, 64]
        layers = []; prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop)]; prev = h
        layers.append(nn.Linear(prev, out))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class PhishScamSenseFusionModel(nn.Module):
    def __init__(self, num_feat=23, nlp_out=128, num_out=64, freeze=True):
        super().__init__()
        self.nlp = NLPBranch(out=nlp_out, freeze=freeze)
        self.mlp = MLPBranch(in_dim=num_feat, out=num_out)
        self.fusion_dim = nlp_out + num_out
    def forward(self, ids, mask, feat):
        return torch.cat([self.nlp(ids, mask), self.mlp(feat)], dim=1)

class URLTokenizer:
    def __init__(self, max_len=128):
        self.tok = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
        self.max_len = max_len
    def tokenize(self, urls):
        return self.tok(urls, padding=True, truncation=True,
                        max_length=self.max_len, return_tensors="pt")

# ── Load checkpoints ──────────────────────────────────────────────────────────
fusion_model = PhishScamSenseFusionModel(num_feat=23)
ckpt_path = CHECKPOINT_DIR / "fusion_model.pt"
if ckpt_path.exists():
    fusion_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print("Fusion model loaded.")
else:
    print("WARNING: Run notebook 04 first to generate checkpoints.")

xgb_path = CHECKPOINT_DIR / "xgb_classifier.pkl"
if xgb_path.exists():
    with open(xgb_path, "rb") as f:
        xgb_clf = pickle.load(f)
    print("XGBoost model loaded.")

fusion_model.eval()
tokenizer = URLTokenizer()

## 6.2 Single URL Inference Pipeline

In [ ]:
RISK_THRESHOLD = 0.6   # minimum confidence to flag as a threat

def predict_url(url: str) -> dict:
    """
    Full 4-class inference pipeline for a single URL.
    Returns the predicted class, per-class probabilities, and risk level.
    """
    tokens = tokenizer.tokenize([url])
    num_feat = torch.tensor([list(extract_url_features(url).values())],
                            dtype=torch.float32).to(device)

    fusion_model.eval()
    with torch.no_grad():
        fused = fusion_model(tokens["input_ids"].to(device),
                             tokens["attention_mask"].to(device),
                             num_feat).cpu().numpy()

    proba     = xgb_clf.predict_proba(fused)[0]   # shape (4,)
    pred_cls  = int(proba.argmax())
    pred_name = CLASS_NAMES[pred_cls]
    confidence = float(proba[pred_cls])
    is_threat  = pred_name != "benign" and confidence >= RISK_THRESHOLD

    return {
        "url":        url,
        "class":      pred_name,
        "confidence": round(confidence, 4),
        "is_threat":  is_threat,
        "risk_level": "HIGH" if confidence > 0.8 else "MEDIUM" if confidence > 0.5 else "LOW",
        "proba":      {name: round(float(p), 4) for name, p in zip(CLASS_NAMES, proba)},
    }


# Test cases: one per class
test_cases = [
    "https://www.google.com",
    "http://paypa1-secure.com/signin/update-billing",
    "amazon.co.uk.security-check.ga",
    "0900259.com",
]

print(f"{'URL':<55} {'Class':<10} {'Conf':<8} {'Threat':<8} Risk")
print("-" * 100)
for url in test_cases:
    r = predict_url(url)
    print(f"{url:<55} {r['class']:<10} {r['confidence']:<8.4f} {str(r['is_threat']):<8} {r['risk_level']}")
    print(f"  probabilities: {r['proba']}")

## 6.3 Batch Inference Pipeline

In [ ]:
import pandas as pd
import time

def predict_batch(urls: list[str], batch_size: int = 32) -> pd.DataFrame:
    """Batch inference with micro-batching. Returns DataFrame with 4-class results."""
    all_results = []
    for i in range(0, len(urls), batch_size):
        batch = urls[i:i+batch_size]
        tokens  = tokenizer.tokenize(batch)
        num_feat = torch.tensor(
            [list(extract_url_features(u).values()) for u in batch],
            dtype=torch.float32,
        )
        fusion_model.eval()
        with torch.no_grad():
            fused = fusion_model(tokens["input_ids"].to(device),
                                 tokens["attention_mask"].to(device),
                                 num_feat.to(device)).cpu().numpy()

        proba_batch = xgb_clf.predict_proba(fused)   # (batch, 4)
        for url, proba in zip(batch, proba_batch):
            pred_cls  = int(proba.argmax())
            pred_name = CLASS_NAMES[pred_cls]
            conf      = float(proba[pred_cls])
            all_results.append({
                "url":        url,
                "class":      pred_name,
                "confidence": round(conf, 4),
                "is_threat":  pred_name != "benign" and conf >= RISK_THRESHOLD,
                "risk_level": "HIGH" if conf > 0.8 else "MEDIUM" if conf > 0.5 else "LOW",
                **{f"p_{name}": round(float(p), 4) for name, p in zip(CLASS_NAMES, proba)},
            })
    return pd.DataFrame(all_results)


# Benchmark
bench_urls = test_cases * 5   # 20 URLs
start = time.time()
batch_df = predict_batch(bench_urls, batch_size=8)
elapsed = time.time() - start

print(f"Batch inference: {len(bench_urls)} URLs in {elapsed:.2f}s  "
      f"({elapsed/len(bench_urls)*1000:.1f} ms/URL)\n")
print(batch_df[["url", "class", "confidence", "is_threat", "risk_level"]].to_string(index=False))

## 6.4 Build Counting Bloom Filter for Browser Extension

The Bloom Filter is the client-side first-pass check inside the browser extension.
- URL **not in filter** → guaranteed safe, no network call needed
- URL **in filter** → call the backend ML service to verify

The filter now covers **all three threat classes** (phishing + malware + spam)
loaded directly from the CIC-Bell-DNS2021 raw files.

In [ ]:
class CountingBloomFilter:
    """
    Counting Bloom Filter matching the TypeScript implementation in
    extension/lib/bloom-filter.ts.

    Supports add, contains, and remove.
    False positive rate ≈ (1 - e^(-k*n/m))^k
    """

    def __init__(self, size: int = 1_000_000, num_hashes: int = 7):
        self.size = size
        self.num_hashes = num_hashes
        self.buckets = [0] * size

    def _hash(self, value: str, seed: int) -> int:
        h = seed
        for char in value:
            h = (h * 31 + ord(char)) & 0xFFFFFFFF
        return h % self.size

    def add(self, value: str):
        for i in range(self.num_hashes):
            idx = self._hash(value, i)
            if self.buckets[idx] < 255:
                self.buckets[idx] += 1

    def contains(self, value: str) -> bool:
        return all(self.buckets[self._hash(value, i)] > 0 for i in range(self.num_hashes))

    def remove(self, value: str):
        if not self.contains(value): return
        for i in range(self.num_hashes):
            idx = self._hash(value, i)
            if self.buckets[idx] > 0:
                self.buckets[idx] -= 1

    def false_positive_rate(self, num_items: int) -> float:
        import math
        return (1 - math.exp(-self.num_hashes * num_items / self.size)) ** self.num_hashes

    def export_data(self) -> list[int]: return self.buckets
    def load_data(self, data: list[int]): self.buckets = list(data)


# ── Load threat URLs from CIC-Bell-DNS2021 (phishing + malware + spam) ───────
print("Loading threat URLs from CIC-Bell-DNS2021…")
urls_all, labels_all = load_cic_bell_dns2021(DATA_DIR, max_benign=0, seed=42)
# max_benign=0 → skip benign entirely, load only threat classes

threat_urls = [u for u, l in zip(urls_all, labels_all) if l != 0]
threat_labels = [l for l in labels_all if l != 0]

from collections import Counter
label_counts = Counter(threat_labels)
for lbl, cnt in sorted(label_counts.items()):
    print(f"  {CLASS_NAMES[lbl]:12s}: {cnt:,}")
print(f"  Total threats : {len(threat_urls):,}")

# Use production-size filter; adjust size for memory constraints
bloom = CountingBloomFilter(size=1_000_000, num_hashes=7)
for url in threat_urls:
    bloom.add(url)

# Verify
print("\nBloom Filter lookup tests:")
for url in threat_urls[:3]:
    print(f"  THREAT  in_filter={bloom.contains(url)}  {url[:60]}")
for url in ["https://www.google.com", "https://github.com", "https://python.org"]:
    print(f"  BENIGN  in_filter={bloom.contains(url)}  {url}")

fpr = bloom.false_positive_rate(len(threat_urls))
print(f"\nFilter size:         {bloom.size:,}")
print(f"Items inserted:      {len(threat_urls):,}")
print(f"False positive rate: {fpr:.6f}  ({fpr*100:.4f}%)")

## 6.5 Export Bloom Filter & Models for Production

In [ ]:
EXPORT_DIR = os.path.join(PROJECT_ROOT, "ml", "exports")
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Export Bloom Filter as JSON (consumed by FastAPI /threats/bloom-filter endpoint)
bloom_export = {
    "filter": bloom.export_data(),
    "size": bloom.size,
    "num_hashes": bloom.num_hashes,
    "version": "0.1.0",
    "num_items": len(known_phishing),
    "false_positive_rate": bloom.false_positive_rate(len(known_phishing)),
}
bloom_path = os.path.join(EXPORT_DIR, "bloom_filter.json")
with open(bloom_path, "w") as f:
    json.dump(bloom_export, f)
print(f"Bloom filter exported: {bloom_path}")

# 2. Export fusion model state dict
fusion_export_path = os.path.join(EXPORT_DIR, "fusion_model.pt")
torch.save(fusion_model.state_dict(), fusion_export_path)
print(f"Fusion model exported: {fusion_export_path}")

# 3. Export XGBoost model
xgb_export_path = os.path.join(EXPORT_DIR, "xgb_classifier.pkl")
with open(xgb_export_path, "wb") as f:
    pickle.dump(xgb_clf, f)
print(f"XGBoost model exported: {xgb_export_path}")

# 4. Export XGBoost as JSON (portable format)
xgb_json_path = os.path.join(EXPORT_DIR, "xgb_classifier.json")
xgb_clf.save_model(xgb_json_path)
print(f"XGBoost JSON exported:  {xgb_json_path}")

print(f"\nAll exports saved to: {EXPORT_DIR}")

## 6.6 End-to-End System Summary

Verify the complete PhishScamSense pipeline works end-to-end.

In [ ]:
def phishscamsense_pipeline(url: str) -> dict:
    """
    Simulates the full two-stage PhishScamSense pipeline:
      Stage 1: Bloom Filter (O(1), client-side, zero network)
      Stage 2: ML 4-class inference (only when Bloom Filter fires)
    """
    bloom_hit = bloom.contains(url)

    if not bloom_hit:
        return {
            "url":       url,
            "stage":     "bloom_filter",
            "class":     "benign",
            "is_threat": False,
            "confidence": 0.0,
            "risk_level": "LOW",
            "note":      "Cleared by Bloom Filter — no ML call needed",
        }

    result = predict_url(url)
    result["stage"] = "ml_inference"
    result["note"]  = "Bloom Filter fired → ML inference performed"
    return result


print("=== PhishScamSense Two-Stage Pipeline Demo ===\n")
demo_urls = [
    "https://www.google.com",              # benign — not in bloom
    "https://python.org",                  # benign — not in bloom
    threat_urls[0],                        # phishing — in bloom
    threat_urls[len(threat_urls)//2],      # malware or spam — in bloom
]

for url in demo_urls:
    r = phishscamsense_pipeline(url)
    status = "THREAT " if r["is_threat"] else "SAFE   "
    print(f"[{r['stage']:<14}]  {status}  class={r['class']:<10}  conf={r['confidence']:.3f}")
    print(f"  URL : {url[:80]}")
    print(f"  Note: {r['note']}\n")

In [ ]:
import matplotlib.pyplot as plt

# Text-based architecture summary
print("=" * 62)
print("  PHISHSCAMSENSE PIPELINE SUMMARY")
print("=" * 62)
print()
print("  Browser Extension (Client-Side)")
print("  ┌───────────────────────────────────────────┐")
print("  │  URL → Counting Bloom Filter              │")
print("  │        O(1) lookup, zero network          │")
print("  │        Covers: phishing + malware + spam  │")
print("  │        If MISS → safe (no API call)       │")
print("  │        If HIT  → call backend API         │")
print("  └────────────────┬──────────────────────────┘")
print("                   │ (only ~5% of URLs)")
print("  ┌────────────────▼──────────────────────────┐")
print("  │  FastAPI Backend  POST /api/v1/predict    │")
print("  └────────────────┬──────────────────────────┘")
print("                   │")
print("  ┌────────────────▼──────────────────────────┐")
print("  │  BentoML Inference Service                │")
print("  │  ┌─────────────┬─────────────────────┐   │")
print("  │  │  NLP Branch │  Numerical Branch   │   │")
print("  │  │  DistilBERT │  23 features        │   │")
print("  │  │  BiLSTM     │  [128, 64] hidden   │   │")
print("  │  │  Attention  │                     │   │")
print("  │  │  → 128d     │  → 64d              │   │")
print("  │  └──────┬──────┴──────────┬──────────┘   │")
print("  │         └──── concat ─────┘              │")
print("  │                192d fused                 │")
print("  │                    ↓                      │")
print("  │         XGBoost 4-class Classifier        │")
print("  │   benign / phishing / malware / spam      │")
print("  └───────────────────────────────────────────┘")
print()
print("  Dataset: CIC-Bell-DNS2021")
print(f"    benign:    ~988k (subsampled to 100k for training)")
print(f"    phishing:    ~16k")
print(f"    malware:     ~27k")
print(f"    spam:         ~8k")
print(f"    Bloom filter: {len(threat_urls):,} threat URLs inserted")
print(f"    FPR:          {bloom.false_positive_rate(len(threat_urls))*100:.4f}%")
print("=" * 62)